ЛР6 ТМО. Канаева Диана ИУ5-62Б

Задачи лабораторной работы:

- обработка пропусков в данных и кодирование категориальных признаков при необходимости
- разделение выборки на обучающую и тестовую с использованием метода train_test_split
- обучение ансамблевой модели группы стекинга
- обучение ансамблевой модели многослойного персептрона
- обучение ансамблевой модели двумя методами на выбор из семейства МГУА
- оценка качества модели с помощью двух подходящих для задачи метрик, сравнение качества полученных моделей

Использован датасет Admission_Predict.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.set(style="ticks")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier, StackingClassifier, RandomForestRegressor, StackingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LinearRegression

In [ ]:
data = pd.read_csv('Admission_Predict.csv', encoding = 'ISO-8859-1')

In [ ]:
data.head()

,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,1,337,118,4,4.5,4.5,9.65,1,0.92
1,2,324,107,4,4.0,4.5,8.87,1,0.76
2,3,316,104,3,3.0,3.5,8.00,1,0.72
3,4,322,110,3,3.5,2.5,8.67,1,0.80
4,5,314,103,2,2.0,3.0,8.21,0,0.65


Типы данных в представленном датасете:

In [ ]:
data.dtypes

Serial No.             int64
GRE Score              int64
TOEFL Score            int64
University Rating      int64
SOP                  float64
LOR                  float64
CGPA                 float64
Research               int64
Chance of Admit      float64
dtype: object

Проверим, есть ли в датасете пропущенные значения:

In [ ]:
data.isnull().sum()

Serial No.           0
GRE Score            0
TOEFL Score          0
University Rating    0
SOP                  0
LOR                  0
CGPA                 0
Research             0
Chance of Admit      0
dtype: int64

В данном датасете нет строк или столбцов, содержащих пропущенные значения.

In [ ]:
data.columns.tolist()

['Serial No.',
 'GRE Score',
 'TOEFL Score',
 'University Rating',
 'SOP',
 'LOR ',
 'CGPA',
 'Research',
 'Chance of Admit ']

Столбец "Serial No." - столбец, включающий в себя индексы строк таблицы. Для дальнейшего анализа он нам не понадобится. Удалим его:

In [ ]:
data = data.drop(columns = 'Serial No.')
data.columns.tolist()

['GRE Score',
 'TOEFL Score',
 'University Rating',
 'SOP',
 'LOR ',
 'CGPA',
 'Research',
 'Chance of Admit ']

In [ ]:
data.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337,118,4,4.5,4.5,9.65,1,0.92
1,324,107,4,4.0,4.5,8.87,1,0.76
2,316,104,3,3.0,3.5,8.00,1,0.72
3,322,110,3,3.5,2.5,8.67,1,0.80
4,314,103,2,2.0,3.0,8.21,0,0.65


Проверим, содержатся ли в данных дубликаты:

In [ ]:
print(data.duplicated())

0      False
1      False
2      False
3      False
4      False
       ...  
395    False
396    False
397    False
398    False
399    False
Length: 400, dtype: bool


In [ ]:
duplicate_rows = data[data.duplicated()]
print(duplicate_rows)

Empty DataFrame
Columns: [GRE Score, TOEFL Score, University Rating, SOP, LOR , CGPA, Research, Chance of Admit ]
Index: []


Дубликатов в данных нет.

С использованием метода train_test_split разделим выборку на обучающую и тестовую:

In [ ]:
data.shape

(400, 8)

In [ ]:
# Признаки без целевой переменной
x = data.drop('Chance of Admit ', axis=1)
x.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research
0,337,118,4,4.5,4.5,9.65,1
1,324,107,4,4.0,4.5,8.87,1
2,316,104,3,3.0,3.5,8.00,1
3,322,110,3,3.5,2.5,8.67,1
4,314,103,2,2.0,3.0,8.21,0


In [ ]:
# Целевая переменная
y = data['Chance of Admit ']
y.head()

0    0.92
1    0.76
2    0.72
3    0.80
4    0.65
Name: Chance of Admit , dtype: float64

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)
#20% данных будут отложены для тестирования, а 80% будут использованы для обучения модели

In [ ]:
# Размер обучающей выборки
x_train.shape, y_train.shape

((320, 7), (320,))

In [ ]:
# Размер тестовой выборки
x_test.shape, y_test.shape

((80, 7), (80,))

Установка модуля gmdh:

In [ ]:
!pip install git+https://github.com/bauman-team/GMDH.git
!pip install gmdh
import gmdh

  Cloning https://github.com/bauman-team/GMDH.git to /tmp/pip-req-build-w8ppjs0g
  Running command git clone --filter=blob:none --quiet https://github.com/bauman-team/GMDH.git /tmp/pip-req-build-w8ppjs0g
  Resolved https://github.com/bauman-team/GMDH.git to commit dddc7b9a76b0930d08ac44bd3d7444473552ddb5
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for gmdh
  Running setup.py clean for gmdh
Failed to build gmdh
ERROR: Could not build wheels for gmdh, which is required to install pyproject.toml-based projects
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.3/875.3 kB 12.9 MB/s eta 0:00:00
  Using cached docstring_inheritance-2.2.0-py3-none-any.whl (24 kB)


In [ ]:
from gmdh import Combi, split_data, Mia
x_train, x_test, y_train, y_test = split_data(x, y)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Модель Combi

In [ ]:
combi_model = gmdh.Combi()
combi_model.fit(x_train, y_train)
y_predicted = combi_model.predict(x_test)

# сравним прогноз с реальным значением
print('y_predicted: ', y_predicted)
print('y_test: ', y_test)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


y_predicted:  [0.68724211 0.6925211  0.66134818 0.5956253  0.62319544 0.8222137
 0.54961717 0.5198961  0.73722787 0.50906314 0.70057623 0.56793068
 0.60087967 0.69484714 0.73129905 0.8329468  0.70835638 0.91578169
 0.74806084 0.73335011 0.6636496  0.73547642 0.62292046 0.61904269
 0.42382695 0.44048887 0.47936669 0.39115294 0.40706516 0.58599383
 0.64486137 0.73520144 0.60598355 0.59347438 0.51749481 0.61729124
 0.75371468 0.57441117 0.47916696 0.61098586 0.76232148 0.87977988
 0.87144977 0.64523622 0.7777819  0.79479406 0.69049466 0.46252966
 0.52369862 0.58424237 0.60663338 0.80425041 0.91087755 0.6784602
 0.49750509 0.49935642 0.44028914 0.46455609 0.49750509 0.65819549
 0.76667396 0.71891436 0.82386529 0.64301005 0.96576746 0.98057806
 0.58757017 0.61358859 0.50898789 0.69262097 0.60673325 0.69902451
 0.80202424 0.71918934 0.83867589 0.78731351 0.80037265 0.8956921
 0.75556601 0.91818298]
y_test:  [0.75 0.73 0.72 0.62 0.67 0.81 0.63 0.69 0.8  0.43 0.8  0.73 0.75 0.71
 0.73 0.83 0.7

Модель MIA

In [ ]:
mia_model = gmdh.Mia()
mia_model.fit(x_train, y_train, polynomial_type=gmdh.PolynomialType.LINEAR)
y_predicted = combi_model.predict(x_test)

# сравним прогноз с реальным значением
print('y_predicted: ', y_predicted)
print('y_test: ', y_test)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


y_predicted:  [0.68724211 0.6925211  0.66134818 0.5956253  0.62319544 0.8222137
 0.54961717 0.5198961  0.73722787 0.50906314 0.70057623 0.56793068
 0.60087967 0.69484714 0.73129905 0.8329468  0.70835638 0.91578169
 0.74806084 0.73335011 0.6636496  0.73547642 0.62292046 0.61904269
 0.42382695 0.44048887 0.47936669 0.39115294 0.40706516 0.58599383
 0.64486137 0.73520144 0.60598355 0.59347438 0.51749481 0.61729124
 0.75371468 0.57441117 0.47916696 0.61098586 0.76232148 0.87977988
 0.87144977 0.64523622 0.7777819  0.79479406 0.69049466 0.46252966
 0.52369862 0.58424237 0.60663338 0.80425041 0.91087755 0.6784602
 0.49750509 0.49935642 0.44028914 0.46455609 0.49750509 0.65819549
 0.76667396 0.71891436 0.82386529 0.64301005 0.96576746 0.98057806
 0.58757017 0.61358859 0.50898789 0.69262097 0.60673325 0.69902451
 0.80202424 0.71918934 0.83867589 0.78731351 0.80037265 0.8956921
 0.75556601 0.91818298]
y_test:  [0.75 0.73 0.72 0.62 0.67 0.81 0.63 0.69 0.8  0.43 0.8  0.73 0.75 0.71
 0.73 0.83 0.7

Оценка качества моделей

In [ ]:
from sklearn.metrics import mean_absolute_error

models = [combi_model, mia_model]
for model in models:
    print(f"{type(model).__name__}:")
    print('\t',f"MAE = {mean_absolute_error(y_test,model.predict(x_test))}")

Combi:
	 MAE = 0.060142259660033726
Mia:
	 MAE = 0.06073088266687936


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
